In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
# Prepare dataset
batch_size = 64
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
 
train_dataset = datasets.MNIST(root='mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)

test_dataset = datasets.MNIST(root='mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

In [3]:
# Define Model
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.channels = channels
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
    
    def forward(self, x):
        y = F.relu(self.conv1(x))
        y = self.conv2(y)
        return F.relu(x + y)

In [4]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)
        self.mp = nn.MaxPool2d(2)
        
        self.rblock1 = ResidualBlock(16)
        self.rblock2 = ResidualBlock(32)
    
        self.fc = nn.Linear(512, 10)
        
    def forward(self, x):
        in_size = x.size(0)
        x = self.mp(F.relu(self.conv1(x)))
        x = self.rblock1(x)
        x = self.mp(F.relu(self.conv2(x)))
        x = self.rblock2(x)
        x = x.view(in_size, -1)
        x = self.fc(x)
        return x
    
model = Net()

In [5]:
# Construct Loss and Optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [6]:
# Train and Test
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        optimizer.zero_grad()
        
        # forward + backward + update
        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
 
        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch + 1, batch_idx + 1, running_loss / 300))
            running_loss = 0.0
            
def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('Accuracy on test set: %d %% ' % (100 * correct / total))

In [7]:
if __name__ == '__main__':
    for epoch in range(10):
        train(epoch)
        test()

[1,   300] loss: 0.532
[1,   600] loss: 0.164
[1,   900] loss: 0.117
Accuracy on test set: 97 % 
[2,   300] loss: 0.095
[2,   600] loss: 0.081
[2,   900] loss: 0.079
Accuracy on test set: 98 % 
[3,   300] loss: 0.062
[3,   600] loss: 0.063
[3,   900] loss: 0.061
Accuracy on test set: 98 % 
[4,   300] loss: 0.054
[4,   600] loss: 0.051
[4,   900] loss: 0.048
Accuracy on test set: 98 % 
[5,   300] loss: 0.042
[5,   600] loss: 0.042
[5,   900] loss: 0.043
Accuracy on test set: 98 % 
[6,   300] loss: 0.036
[6,   600] loss: 0.037
[6,   900] loss: 0.038
Accuracy on test set: 98 % 
[7,   300] loss: 0.031
[7,   600] loss: 0.033
[7,   900] loss: 0.033
Accuracy on test set: 98 % 
[8,   300] loss: 0.031
[8,   600] loss: 0.030
[8,   900] loss: 0.028
Accuracy on test set: 98 % 
[9,   300] loss: 0.025
[9,   600] loss: 0.025
[9,   900] loss: 0.027
Accuracy on test set: 98 % 
[10,   300] loss: 0.022
[10,   600] loss: 0.025
[10,   900] loss: 0.023
Accuracy on test set: 99 % 
